# 📖 Notebook 1: Cache Partitioning Strategies

When your data no longer fits on one machine, you need to **split it across multiple nodes**.
This is called **partitioning** (or **sharding**). The key question is:

> Given a key, which node should store it?

The answer to that question has massive implications for performance, balance, and what
happens when you add or remove nodes.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why a single cache node isn't enough at scale
- How **modulo hashing** works and why it breaks on resize
- How **range partitioning** works and its hot-spot problem
- Why **consistent hashing** is the industry standard (deep dive in Notebook 3)
- How to measure key distribution across nodes

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/distributed-cache
docker-compose up -d
```

### RedisInsight (Redis GUI)
- **URL**: http://localhost:5540
- Add each node: `localhost:6381`, `localhost:6382`, `localhost:6383`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import redis
import hashlib
import time

# Our 3-node cache cluster — each is a plain Redis instance on a different port
NODES = {
    "node-1": redis.Redis(host="localhost", port=6381, decode_responses=True),
    "node-2": redis.Redis(host="localhost", port=6382, decode_responses=True),
    "node-3": redis.Redis(host="localhost", port=6383, decode_responses=True),
}

NODE_LIST = list(NODES.keys())   # ["node-1", "node-2", "node-3"]

# Verify all nodes are up
for name, client in NODES.items():
    try:
        client.ping()
        print(f"✅ {name} is up (port {client.connection_pool.connection_kwargs['port']})")
    except Exception as e:
        print(f"❌ {name} failed: {e}")
        print("   Run: docker-compose up -d")

## 📏 Why One Node Isn't Enough

Imagine you're building a cache for an e-commerce site.

- You have **1 million products** with an average cached value of **1 KB**.
- That's **~1 GB** of cache data — one node can handle that.

Now your site grows:
- **100 million products**, each **10 KB** → **1 TB** of cache data.
- **100,000 requests/second** hitting the cache.

A single server with 32 GB RAM can hold maybe **24 GB** of cache data (rest goes to the OS
and Redis overhead). You'd need **~43 nodes** just for storage!

| Requirement | Single Node | 50-Node Cluster |
|-------------|-------------|------------------|
| Storage     | 24 GB       | 1.2 TB           |
| Throughput  | ~20k req/s  | ~1M req/s        |
| Availability| One crash = down | One crash = 2% lost |

**The question becomes: given a key like `product:42`, which of our 50 nodes should store it?**

---

## Strategy 1: Modulo Hashing

The simplest approach: hash the key, take the remainder when divided by the number of nodes.

```
node_index = hash(key) % number_of_nodes
```

For example, with 3 nodes:
- `hash("product:42") % 3 = 1` → goes to node-2
- `hash("product:99") % 3 = 0` → goes to node-1
- `hash("product:7")  % 3 = 2` → goes to node-3

In [ ]:
def hash_key(key: str) -> int:
    """Create a numeric hash from a string key using MD5."""
    return int(hashlib.md5(key.encode()).hexdigest(), 16)


def modulo_partition(key: str, num_nodes: int) -> int:
    """Pick a node index using modulo hashing."""
    return hash_key(key) % num_nodes


# Demo: which node does each product go to?
print("🗂️  Modulo Hashing with 3 nodes")
print("=" * 50)
for i in range(10):
    key = f"product:{i}"
    node_idx = modulo_partition(key, 3)
    print(f"  {key:<15} → hash % 3 = {node_idx} → {NODE_LIST[node_idx]}")

In [ ]:
# Let's actually store data across our 3 Redis nodes using modulo hashing

# Clear all nodes first
for client in NODES.values():
    client.flushall()

# Insert 1000 products using modulo routing
for i in range(1000):
    key = f"product:{i}"
    node_idx = modulo_partition(key, 3)
    node_name = NODE_LIST[node_idx]
    NODES[node_name].set(key, f"Product #{i} data — price: ${i * 1.5:.2f}")

# Count keys per node
print("📊 Key distribution across nodes (1000 keys, modulo hashing):")
print()
for name, client in NODES.items():
    count = client.dbsize()
    bar = "█" * (count // 10)
    print(f"  {name}: {count:>4} keys  {bar}")

print()
print("✅ Keys are roughly evenly distributed — modulo hashing works great!")
print("   ...until we change the number of nodes.")

### 💥 The Resize Problem

What happens when we go from 3 nodes to 4? Let's see how many keys would need to move.

In [ ]:
# Simulate: what happens when we go from 3 nodes to 4?

keys_that_moved = 0
total_keys = 1000

for i in range(total_keys):
    key = f"product:{i}"
    old_node = modulo_partition(key, 3)  # hash % 3
    new_node = modulo_partition(key, 4)  # hash % 4
    if old_node != new_node:
        keys_that_moved += 1

pct = keys_that_moved / total_keys * 100

print(f"💥 Resizing from 3 → 4 nodes with modulo hashing")
print(f"   Keys that need to move: {keys_that_moved} / {total_keys} ({pct:.1f}%)")
print()
print(f"   That means ~{pct:.0f}% of your cache is INVALIDATED on resize!")
print(f"   All those keys will cause cache misses → thundering herd to the database.")
print()
print("   This is why modulo hashing is dangerous in production.")
print("   With consistent hashing (Notebook 3), only ~1/N keys move.")

In [ ]:
# Let's visualize the impact across different resize scenarios

print("📊 Percentage of keys that move during resize (modulo hashing)")
print("=" * 60)
print(f"{'Resize':<15} {'Keys Moved':>12} {'Percentage':>12}  Impact")
print("-" * 60)

scenarios = [
    (3, 4,  "Add 1 node"),
    (3, 5,  "Add 2 nodes"),
    (3, 6,  "Double cluster"),
    (3, 2,  "Remove 1 node"),
    (10, 11, "10→11 nodes"),
    (50, 51, "50→51 nodes"),
]

for old_n, new_n, label in scenarios:
    moved = sum(
        1 for i in range(10000)
        if modulo_partition(f"key:{i}", old_n) != modulo_partition(f"key:{i}", new_n)
    )
    pct = moved / 10000 * 100
    severity = "🔴" if pct > 60 else "🟡" if pct > 30 else "🟢"
    print(f"  {old_n}→{new_n} ({label:<14}) {moved:>6}/10000  {pct:>8.1f}%    {severity}")

print()
print("🔑 Key insight: modulo hashing moves ~75% of keys on ANY resize!")
print("   Consistent hashing (Notebook 3) moves only ~1/N keys.")

---

## Strategy 2: Range Partitioning

Instead of hashing, assign **key ranges** to each node:

| Node   | Key Range         |
|--------|-------------------|
| node-1 | `a` – `h`         |
| node-2 | `i` – `p`         |
| node-3 | `q` – `z`         |

This is how some databases (like HBase) partition data. It's great for **range queries**
("give me all products from A to D"), but terrible for even distribution.

In [ ]:
def range_partition(key: str, num_nodes: int) -> int:
    """Pick a node based on the first character of the key."""
    first_char = key[0].lower()
    # Divide the alphabet into roughly equal ranges
    alphabet_pos = ord(first_char) - ord('a')
    chars_per_node = 26 / num_nodes
    return min(int(alphabet_pos / chars_per_node), num_nodes - 1)


# Demo with user names (natural language keys have uneven letter distribution)
users = [
    "alice", "bob", "charlie", "david", "emma", "frank",
    "grace", "henry", "ivy", "jack", "karen", "liam",
    "mia", "noah", "olivia", "peter", "quinn", "rachel",
    "sam", "tom", "uma", "victor", "wendy", "xander",
]

# Count per node
node_counts = {name: 0 for name in NODE_LIST}
for user in users:
    idx = range_partition(user, 3)
    node_counts[NODE_LIST[idx]] += 1

print("📊 Range partitioning with user names (a-h → node-1, i-p → node-2, q-z → node-3)")
print()
for name, count in node_counts.items():
    bar = "█" * count
    print(f"  {name}: {count:>3} users  {bar}")

print()
print("⚠️  Notice the uneven distribution!")
print("   English names aren't evenly spread across the alphabet.")
print("   Letters like 'j', 's', 'm' are way more common than 'x', 'q', 'z'.")

In [ ]:
# Let's prove the hot-spot problem with a larger dataset
import random
random.seed(42)

# Simulate 10,000 product names starting with common letters
# In English, letters have very different frequencies
letter_weights = {
    'a': 8, 'b': 6, 'c': 7, 'd': 5, 'e': 4, 'f': 3, 'g': 3, 'h': 4,
    'i': 3, 'j': 2, 'k': 2, 'l': 4, 'm': 5, 'n': 3, 'o': 3, 'p': 5,
    'q': 1, 'r': 4, 's': 8, 't': 6, 'u': 2, 'v': 2, 'w': 3, 'x': 1,
    'y': 1, 'z': 1
}

letters = list(letter_weights.keys())
weights = list(letter_weights.values())
keys = [random.choices(letters, weights=weights, k=1)[0] + f"_product:{i}" for i in range(10000)]

range_counts = [0, 0, 0]
modulo_counts = [0, 0, 0]

for key in keys:
    range_counts[range_partition(key, 3)] += 1
    modulo_counts[modulo_partition(key, 3)] += 1

print("📊 10,000 keys — Range vs Modulo distribution")
print("=" * 55)
print(f"{'Node':<10} {'Range':>10} {'Modulo':>10}   {'Ideal':>10}")
print("-" * 55)
for i in range(3):
    print(f"  {NODE_LIST[i]:<10} {range_counts[i]:>8}   {modulo_counts[i]:>8}   {10000//3:>8}")

print()
print("🔑 Range partitioning creates hot spots with natural-language keys.")
print("   Modulo hashing distributes evenly, but breaks on resize.")
print("   We need the best of both worlds → Consistent Hashing (Notebook 3).")

---

## Strategy 3: Consistent Hashing (Preview)

Consistent hashing solves the resize problem by mapping both **nodes and keys** onto a
circular ring (0 to 2³²). Each key goes to the next node clockwise on the ring.

When a node is added, only the keys between the new node and its predecessor need to move —
roughly **1/N** of all keys instead of **~75%**.

We'll do a full deep-dive in **Notebook 3**. Here's a quick preview to see the difference.

In [ ]:
import bisect

class SimpleHashRing:
    """Minimal consistent hash ring — just enough to show the concept."""

    def __init__(self, nodes: list[str]):
        self.ring = []          # sorted list of (position, node_name)
        self.positions = []     # sorted positions only (for bisect)
        for node in nodes:
            pos = hash_key(node) % (2**32)
            self.ring.append((pos, node))
            self.positions.append(pos)
        # Sort by position on the ring
        paired = sorted(zip(self.positions, [r[1] for r in self.ring]))
        self.positions = [p[0] for p in paired]
        self.ring = paired

    def get_node(self, key: str) -> str:
        """Find the next node clockwise from the key's position."""
        pos = hash_key(key) % (2**32)
        idx = bisect.bisect_right(self.positions, pos)
        if idx >= len(self.positions):
            idx = 0  # wrap around the ring
        return self.ring[idx][1]


# Compare: add a 4th node — how many keys move?
ring_3 = SimpleHashRing(["node-1", "node-2", "node-3"])
ring_4 = SimpleHashRing(["node-1", "node-2", "node-3", "node-4"])

total = 10000
modulo_moved = 0
ring_moved = 0

for i in range(total):
    key = f"product:{i}"
    # Modulo approach
    if modulo_partition(key, 3) != modulo_partition(key, 4):
        modulo_moved += 1
    # Consistent hashing approach
    if ring_3.get_node(key) != ring_4.get_node(key):
        ring_moved += 1

print("📊 Keys moved when adding 1 node (3 → 4 nodes)")
print("=" * 55)
print(f"  Modulo hashing:      {modulo_moved:>5} / {total} ({modulo_moved/total*100:.1f}%)  🔴")
print(f"  Consistent hashing:  {ring_moved:>5} / {total} ({ring_moved/total*100:.1f}%)  🟢")
print()
print(f"  Consistent hashing moved {modulo_moved/max(ring_moved,1):.0f}× fewer keys!")
print()
print("🔑 This is why every major distributed cache uses consistent hashing.")
print("   Deep dive in Notebook 3!")

---

## 🧪 Try It Yourself

1. **Change the number of keys**: Try 100, 10,000, or 100,000 keys. Does modulo
   hashing distribute them evenly? What about the resize problem?

2. **Real-world keys**: Replace `product:{i}` with realistic keys like UUIDs,
   email addresses, or URLs. Does the distribution change?

3. **Open RedisInsight** at http://localhost:5540 and add the 3 Redis nodes.
   Can you see the keys on each node? Do the counts match what Python reported?

## 📝 Key Takeaways

| Strategy | Even Distribution | Resize Cost | Range Queries | Use Case |
|----------|------------------|-------------|---------------|----------|
| Modulo   | ✅ Great          | ❌ ~75% move | ❌ No          | Fixed clusters |
| Range    | ❌ Hot spots      | ✅ Easy      | ✅ Yes         | Ordered data |
| Consistent Hash | ✅ Great   | ✅ ~1/N move | ❌ No          | **Production caches** |

## ➡️ Next: Notebook 2 — Cache Coherence & Invalidation

Now that we know *where* to put data, the next question is:  
**How do we keep it consistent when we have copies on multiple nodes?**

In [ ]:
# Cleanup: remove all keys we created
for client in NODES.values():
    client.flushall()
print("🧹 All nodes flushed.")